In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("dataset_2191_sleep.csv")

# Check the column names
print(df.columns)

In [ ]:
# Select the relevant variables and remove missing values
data = df[['total_sleep', 'max_life_span']].dropna()

x = data['total_sleep'].to_numpy()
y = data['max_life_span'].to_numpy()

print("Sample size:", len(x))

In [ ]:
# Define significance level
alpha = 0.05

# Perform linear regression
results = stats.linregress(x, y)

slope = results.slope
intercept = results.intercept
p_value = results.pvalue
std_error = results.stderr
r_value = results.rvalue

# Calculate test statistic
t_stat = slope / std_error

print("Sample size:", len(x))
print("Slope:", slope)
print("Intercept:", intercept)
print("Standard error:", std_error)
print("t-statistic:", t_stat)
print("Degrees of freedom:", len(x) - 2)
print("p-value:", p_value)
print("Correlation coefficient:", r_value)
print("R-squared:", r_value**2)

# Hypothesis test conclusion
if p_value < alpha:
    print("Reject the null hypothesis.")
    print("There is a statistically significant linear relationship.")
else:
    print("Fail to reject the null hypothesis.")
    print("There is insufficient evidence of a linear relationship.")

In [ ]:
def linear_regression(x, y):
    results = stats.linregress(x, y)
    return results.slope, results.intercept

a, b = linear_regression(x, y)

res = stats.bootstrap(
    (x, y),
    linear_regression,
    confidence_level=0.95,
    paired=True,
    n_resamples=10000,
    random_state=42
)

low = res.confidence_interval.low
high = res.confidence_interval.high

print("Slope:", a)
print("95% CI:", [low[0], high[0]])

print("Intercept:", b)
print("95% CI:", [low[1], high[1]])

print("p-value:", p_value)

In [ ]:
x_line = np.linspace(min(x), max(x), 100)
y_line = intercept + slope * x_line

plt.figure(figsize=(8, 5))

plt.scatter(x, y, c='k', label='Observed data')

plt.plot(
    x_line,
    y_line,
    color='red',
    label=f'y = {slope:.3f}x + {intercept:.3f}'
)

plt.xlabel('Total sleep duration (hours)')
plt.ylabel('Maximum lifespan (years)')

plt.title(
    f'Linear Regression\n'
    f'R² = {r_value**2:.3f}, p = {p_value:.4g}'
)

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
predictions = intercept + slope * x
residuals = y - predictions

plt.figure(figsize=(8, 5))

plt.scatter(predictions, residuals, c='k')
plt.axhline(0, color='red', linestyle='--')

plt.xlabel('Predicted maximum lifespan')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')

plt.show()

# Normal Q-Q plot
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Normal Q-Q Plot")
plt.show()

# Shapiro-Wilk normality test
print(stats.shapiro(residuals))